In [10]:
import psutil
import time

def print_system_resources():
    cpu_pct  = psutil.cpu_percent(interval=1)  # % over last 1 second
    ram      = psutil.virtual_memory()
    swap     = psutil.swap_memory()
    disk_io  = psutil.disk_io_counters()

    print(f'CPU Usage       : {cpu_pct:.1f}%')
    print(f'RAM In Use      : {ram.used  / 1e9:.2f} GB  /  {ram.total / 1e9:.2f} GB  ({ram.percent:.1f}%)')
    print(f'RAM Available   : {ram.available / 1e9:.2f} GB')
    print(f'Swap In Use     : {swap.used / 1e9:.2f} GB  ({swap.percent:.1f}%)')

# Call at each training epoch
for epoch in range(10):
    print(f'\n--- Epoch {epoch+1} ---')
    print_system_resources()
    # ... training code here ...


--- Epoch 1 ---
CPU Usage       : 2.0%
RAM In Use      : 0.92 GB  /  13.61 GB  (9.0%)
RAM Available   : 12.39 GB
Swap In Use     : 0.00 GB  (0.0%)

--- Epoch 2 ---
CPU Usage       : 2.0%
RAM In Use      : 0.92 GB  /  13.61 GB  (9.0%)
RAM Available   : 12.39 GB
Swap In Use     : 0.00 GB  (0.0%)

--- Epoch 3 ---
CPU Usage       : 2.0%
RAM In Use      : 0.92 GB  /  13.61 GB  (9.0%)
RAM Available   : 12.39 GB
Swap In Use     : 0.00 GB  (0.0%)

--- Epoch 4 ---
CPU Usage       : 13.6%
RAM In Use      : 0.90 GB  /  13.61 GB  (8.8%)
RAM Available   : 12.41 GB
Swap In Use     : 0.00 GB  (0.0%)

--- Epoch 5 ---
CPU Usage       : 53.3%
RAM In Use      : 0.89 GB  /  13.61 GB  (8.8%)
RAM Available   : 12.42 GB
Swap In Use     : 0.00 GB  (0.0%)

--- Epoch 6 ---
CPU Usage       : 13.1%
RAM In Use      : 0.91 GB  /  13.61 GB  (8.9%)
RAM Available   : 12.40 GB
Swap In Use     : 0.00 GB  (0.0%)

--- Epoch 7 ---
CPU Usage       : 2.0%
RAM In Use      : 0.91 GB  /  13.61 GB  (8.9%)
RAM Available   : 12.4

In [29]:
import torch

def print_gpu_resources():
    if not torch.cuda.is_available():
        print('No CUDA GPU available.')
        return

    for i in range(torch.cuda.device_count()):
        props     = torch.cuda.get_device_properties(i)
        mem_used  = torch.cuda.memory_allocated(i)    # bytes in active tensors
        mem_res   = torch.cuda.memory_reserved(i)     # bytes reserved by allocator
        mem_total = props.total_memory

        print(f'GPU {i}: {props.name}')
        print(f'  VRAM Allocated : {mem_used  / 1e9:.2f} GB')
        print(f'  VRAM Reserved  : {mem_res   / 1e9:.2f} GB')
        print(f'  VRAM Total     : {mem_total / 1e9:.2f} GB')
        print(f'  Utilisation    : {mem_used / mem_total * 100:.1f}%')

# Call after each training step
print_gpu_resources()

GPU 0: Tesla T4
  VRAM Allocated : 0.08 GB
  VRAM Reserved  : 0.41 GB
  VRAM Total     : 15.64 GB
  Utilisation    : 0.5%


In [18]:
# vram_inference_test.py
import torch
from torchvision.models import resnet18, ResNet18_Weights
import psutil

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {device}')

def show_memory():
    ram = psutil.virtual_memory()
    print(f'  RAM  : {ram.used/1e9:.2f} GB used  ({ram.percent:.1f}%)')
    if device == 'cuda':
        print(f'  VRAM : {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')
        print(f'  VRAM : {torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

print('Before loading model:')
show_memory()

# Load a pre-trained ResNet-18 (11 million parameters, ~44 MB at FP32)
model = resnet18(weights=ResNet18_Weights.DEFAULT).to(device)

model.eval()
print('After loading model to device:')
show_memory()

# Create a batch of 32 random images (simulating input data)
batch = torch.randn(32, 3, 224, 224).to(device)

print('After loading batch to device:')
show_memory()

# Run inference (no gradients needed)
with torch.no_grad():
    output = model(batch)

print('After inference:')
show_memory()
print(f'Output shape: {output.shape}')  # Expected: torch.Size([32, 1000])

Running on: cuda
Before loading model:
  RAM  : 1.75 GB used  (15.1%)
  VRAM : 0.00 GB allocated
  VRAM : 0.00 GB reserved
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 185MB/s]


After loading model to device:
  RAM  : 1.91 GB used  (16.4%)
  VRAM : 0.05 GB allocated
  VRAM : 0.07 GB reserved
After loading batch to device:
  RAM  : 1.91 GB used  (16.4%)
  VRAM : 0.07 GB allocated
  VRAM : 0.09 GB reserved
After inference:
  RAM  : 2.02 GB used  (17.2%)
  VRAM : 0.08 GB allocated
  VRAM : 0.41 GB reserved
Output shape: torch.Size([32, 1000])


In [31]:
# dataset_memory_test.py import numpy as np import psutil   def ram_gb(): return psutil.virtual_memory().used / 1e9   print(f'RAM before : {ram_gb():.2f} GB')   # Simulate loading 10,000 images at 224x224 pixels, 3 colour channels # Each pixel value is a float32 (4 bytes) # Total: 10000 x 224 x 224 x 3 x 4 bytes = 6.02 GB num_images  = 10_000 height, width, channels = 224, 224, 3   print(f'Allocating simulated dataset ({num_images} images at {height}x{width}x{channels})...') dataset = np.zeros((num_images, height, width, channels), dtype=np.float32)   print(f'RAM after  : {ram_gb():.2f} GB') expected_gb = num_images * height * width * channels * 4 / 1e9 print(f'Expected increase: {expected_gb:.2f} GB') print(f'Dataset array size: {dataset.nbytes / 1e9:.2f} GB')   input('Press Enter to release dataset...') del dataset import gc; gc.collect() print(f'RAM after release : {ram_gb():.2f} GB')

In [27]:
# memory_test.py # Demonstrates deliberate RAM allocation and measurement

import psutil
import time

def ram_used_gb():
    return psutil.virtual_memory().used / 1e9

print(f'RAM in use before allocation : {ram_used_gb():.2f} GB')

# Allocate a list of 100 million integers
# Python interns the integer 0, so only one integer object exists.
# Memory cost is 100 million 8-byte pointers in the list structure.
# Total: ~800 MB of RAM

before = ram_used_gb()
large_list = [0] * 100_000_000

after = ram_used_gb()

print(f'RAM in use after allocation  : {after:.2f} GB')
print(f'Change: +{after - before:.2f} GB')

input('RAM is allocated. Open Task Manager now and observe. Press Enter to free memory.')

# Deleting the variable tells Python's garbage collector to free the memory
del large_list
import gc
gc.collect()  # force immediate garbage collection

print(f'RAM in use after deletion    : {ram_used_gb():.2f} GB')
print('Memory freed successfully.')

RAM in use before allocation : 2.21 GB
RAM in use after allocation  : 2.98 GB
Change: +0.77 GB
RAM is allocated. Open Task Manager now and observe. Press Enter to free memory.
RAM in use after deletion    : 2.18 GB
Memory freed successfully.


In [33]:
# Single snapshot of GPU status nvidia-smi   # Continuous monitoring — refresh every 1 second (press Ctrl+C to stop) nvidia-smi -l 1   # Compact loop format showing only key metrics every second nvidia-smi --querygpu=name,utilization.gpu,utilization.memory,memory.used,memory.t otal,temperature.gpu --format=csv -l 1